In [ ]:
import torch
import torch.nn as nn
from torchvision.io import read_video
from PIL import Image
import numpy as np
import os

from VideoTransformer import VideoTransformer
from AspectRatioPad import AspectRatioPad
from torchvision import models

# 1. 前処理クラス (学習時と同じものを用意)
class Predictor:
    def __init__(self, model_path, class_names, n_seconds=2, L_frames=8, device="cpu"):
        self.device = torch.device(device)
        self.n_seconds = n_seconds
        self.L_frames = L_frames
        self.class_names = class_names
        
        # モデルの構築と重みのロード
        self.model = VideoTransformer(num_classes=len(class_names)).to(self.device)
        self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        self.model.eval()
        
        # 前処理
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        self.preprocess = weights.transforms()
        self.padder = AspectRatioPad(size=(224, 224))

    def predict(self, video_path):
        # 2. 動画の読み込みとサンプリング
        video, _, _ = read_video(video_path, start_pts=0, end_pts=self.n_seconds, pts_unit='sec')
        total_frames = video.shape[0]
        indices = torch.linspace(0, total_frames - 1, steps=self.L_frames).long()
        sampled_video = video[indices]

        # 3. 各フレームの前処理
        processed_frames = []
        for frame in sampled_video:
            img = Image.fromarray(frame.numpy())
            img = self.padder(img)
            img = self.preprocess(img)
            processed_frames.append(img)
        
        # (L, C, H, W) -> (1, L, C, H, W) バッチ次元を追加
        input_tensor = torch.stack(processed_frames).unsqueeze(0).to(self.device)

        # 4. 推論
        with torch.no_grad():
            outputs = self.model(input_tensor)
            # 確率に変換 (Softmax)
            probs = torch.nn.functional.softmax(outputs[0], dim=0)
            conf, pred_idx = torch.max(probs, 0)

        return self.class_names[pred_idx], conf.item()

In [ ]:
# --- 実行 ---
# クラス名のリスト (フォルダ名の順序と一致させること)
classes = ['background','alert', 'hungry', 'log_time_no_see', 'miss']

out_dir="output"
latest_path = os.path.join(out_dir, "latest_model.pth")

predictor = Predictor(
    model_path=latest_path, 
    class_names=classes,
    n_seconds=4,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

In [ ]:
datasets="dataset_h264/alert"
mp4 = os.path.join(datasets, "5nDkbL55-tY_trim1.mp4")
label, confidence = predictor.predict(mp4)
print(f"判定結果: {label} (確信度: {confidence:.2%})")
